In [1]:
# pip install pinecone-client scikit-learn beautifulsoup4
from pathlib import Path
from typing import List, Dict, Any
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_KEY")

In [11]:
# Loading the prompt
# Old Code for using Chain... quick refersher without RAG
# --- set the path OUTSIDE the functions ---
PROMPT_PATH = r"../prompt.yaml"   # e.g., "/home/user/project/prompt.yaml"

# --- tiny YAML loader (file only) ---
def load_prompt(path=PROMPT_PATH):
    """Load prompt config strictly from a YAML file."""
    from pathlib import Path
    import yaml  # pip install pyyaml

    text = Path(path).read_text(encoding="utf-8")
    cfg = yaml.safe_load(text) or {}
    
    return cfg

prompt = load_prompt(PROMPT_PATH)
prompt = prompt['template']
prompt

'Use the sources below to answer concisely in a casual coffee-chat tone.\nIf unsure, say so briefly.\nKeep it to ~120–150 words\n\nsources:\n{sources}\n\nquery:\n{query}'

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=gemini_api_key
)

# prompt template from YAML
prompt_template = PromptTemplate(
    template=prompt,
    input_variables=["query", "sources"]
)

# LCEL chain
chain = prompt_template | llm | StrOutputParser()

user_question = "Tell me about coffee"

chain.invoke({
    "query": user_question,
    "sources": "No sources available right now"
})


"Hey there! You're asking about coffee, which is a fantastic topic, honestly! I'd love to brew up a whole response for you right now. But, I've got a bit of a snag here: looking at the sources you've provided for me to use, it explicitly says, 'No sources available right now.'\n\nBecause my instructions are to *only* use those sources to answer, and there aren't any, I'm completely in the dark on this one! I genuinely can't tell you anything about coffee from the materials I'm supposed to reference. It's like having an empty coffee pot when you're craving a fresh cup – nothing to pour from! So, unfortunately, I'm unable to give you the information you're looking for based on what's been supplied. Wish I could spill the beans, but there are no beans in my source cupboard!"

In [ ]:
# Chain's are langchain runnable sequence 
# runnable parallel
# runnable lambda 

In [6]:
def get_upper(d):
    return str(d).upper()

def get_greeting(name):
    return f"hello {name}"

In [8]:
upper_name = get_upper('kshitij')
upper_name

'KSHITIJ'

In [9]:
get_greeting(upper_name)

'hello KSHITIJ'

In [10]:
from langchain_core.runnables import RunnableLambda

chain = RunnableLambda(get_upper) | RunnableLambda(get_greeting)

chain.invoke("kshitij")

'hello KSHITIJ'

# User retrival part using chains and runnable lambda 

Steps for Query Retrival and LLM O/P Generation

User query --> Creating embbedding (dense & sparse) --> Retrive data from Vecotor DB --> Add to prompt --> Pass it to LLM

In [15]:
# Required Functions
import numpy as np

INDEX_NAME = "vector-hybrid-v1"

pc = Pinecone(api_key=pinecone_api_key)

index = pc.Index(INDEX_NAME)

model_emb = SentenceTransformer('all-MiniLM-L6-v2')
vectorizer = TfidfVectorizer()

def hybrid_query(query_text: str, alpha: float = 0.5, top_k: int = 5):
    """Perform a hybrid search (dense + sparse) and return top-k doc_id, score, preview."""
    # Dense embedding scaled by (1 - alpha)
    q_dense = model_emb.encode([query_text], normalize_embeddings=True)[0]
    q_dense = (np.asarray(q_dense, dtype=float) * (1.0 - alpha)).tolist()

    # Sparse embedding scaled by alpha
    # q_sparse_csr = vectorizer.transform([query_text]).tocoo()
    # q_sparse = (
    #     {"indices": [0], "values": [0.0]}
    #     if q_sparse_csr.nnz == 0
    #     else {
    #         "indices": q_sparse_csr.col.tolist(),
    #         "values": (q_sparse_csr.data.astype(float) * alpha).tolist(),
    #     }
    # )

    # Query Pinecone
    res = index.query(
        vector=q_dense,
        #sparse_vector=q_sparse,
        top_k=top_k,
        include_metadata=True
    )

    # Format concise output
    results = []
    #print(f"\nQuery: {query_text!r}  | alpha={alpha}")
    #print("-" * 70)
    for m in res.get("matches", []):
        doc_id = m["id"]
        score = round(m["score"], 4)
        text = (m.get("metadata", {}) or {}).get("text", "")
        preview = " ".join(text.split()[:100])
        results.append({"id": doc_id, "score": score, "preview": preview})
        #print(f"Doc ID: {doc_id}  |  Score: {score}\nPreview: {preview}\n")
    #print("-" * 70)

    return results

def create_prompt(query, results):
    cfg = load_prompt()

    source_text = "\n\n".join(
        r["preview"] for r in results if r.get("preview")
    ) or "No sources found."

    # fill values into the prompt
    prompt = cfg["template"].format(
        query=query,
        sources=source_text
    )

    return prompt


# Old Code for using Chain... quick refersher without RAG

# --- set the path OUTSIDE the functions ---
PROMPT_PATH = r"prompt.yaml"   # e.g., "/home/user/project/prompt.yaml"

# --- tiny YAML loader (file only) ---
def load_prompt(path=PROMPT_PATH):
    """Load prompt config strictly from a YAML file."""
    from pathlib import Path
    import yaml  # pip install pyyaml

    text = Path(path).read_text(encoding="utf-8")
    cfg = yaml.safe_load(text) or {}
    
    return cfg

NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '40936bd0211cb9aa5913b46b0132ae68', 'date': 'Sat, 20 Dec 2025 13:18:44 GMT', 'server': 'Google Frontend', 'Content-Length': '91', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource vector-hybrid-v1 not found"},"status":404}


In [14]:
user_q = "Tell me about healthy coffee"
results = hybrid_query(user_q)

In [15]:
create_prompt(user_q, results)

"Use the sources below to answer concisely in a casual coffee-chat tone.\nKeep it to ~120–150 words. If unsure, say so briefly.\n\nSources:\nBeaten Coffee (Phenti Hui Coffee) Updated on August 17, 2025 Disclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines. Overview\n\nOverview Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance. Ingredients (Base & Optional) 1 cup milk of choice (dairy or plant-based) ½–1 tsp instant coffee or a single espresso shot ¼–½ tsp ashwagandha powder (culinary grade) ¼ tsp cinnamon or cardamom (optional)\n\nOverview Cold brew extracts coffee slowly

In [13]:
from langchain_core.runnables import RunnableLambda

chain = RunnableLambda(hybrid_query) 

user_q = "Tell me about healthy coffee"

chain.invoke(user_q)

[{'id': 'af089355-9817-4c4e-bb46-765d583f0a5b',
  'score': 0.9073,
  'preview': 'Beaten Coffee (Phenti Hui Coffee) Updated on August 17, 2025 Disclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines. Overview'},
 {'id': '7a43c90a-8347-4de4-b815-9e83fe26744c',
  'score': 0.8942,
  'preview': 'Overview Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance. Ingredients (Base & Optional) 1 cup milk of choice (dairy or plant-based) ½–1 tsp instant coffee or a single espresso shot ¼–½ tsp ashwagandha powder (culinary grade) ¼ tsp cinnamon or cardamom (optional)'},
 {'id': 'd3e67194-41

In [17]:
from langchain_core.runnables import RunnableLambda

chain = RunnableLambda(hybrid_query) | RunnableLambda(lambda results:create_prompt(user_q, results))

user_q = "Tell me about healthy coffee"

chain.invoke(user_q)

"Use the sources below to answer concisely in a casual coffee-chat tone.\nKeep it to ~120–150 words. If unsure, say so briefly.\n\nSources:\nBeaten Coffee (Phenti Hui Coffee) Updated on August 17, 2025 Disclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines. Overview\n\nOverview Ashwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance. Ingredients (Base & Optional) 1 cup milk of choice (dairy or plant-based) ½–1 tsp instant coffee or a single espresso shot ¼–½ tsp ashwagandha powder (culinary grade) ¼ tsp cinnamon or cardamom (optional)\n\nOverview Cold brew extracts coffee slowly

In [18]:
# Basic RAG
from langchain_core.runnables import RunnableLambda

chain = (
    RunnableLambda(hybrid_query)  # Search from vector DB and retreving the documents
  | RunnableLambda(lambda results:create_prompt(user_q, results)) # Augument to the prompt
  | llm # Passing prompt to LLM
  | StrOutputParser()  # Getting our Output
)
user_q = "Tell me about healthy coffee"

chain.invoke(user_q)

'Hey there! So, based on these notes, when we talk about "healthy coffee," it\'s a bit tricky because the sources don\'t really dive into medical advice, actually. They even have a disclaimer telling folks to chat with a professional before trying new herbs or routines, especially if they have health conditions.\n\nHowever, they do mention a couple of interesting takes. There\'s ashwagandha coffee, which mixes roasted coffee with powdered ashwagandha, an herb "widely used in Indian traditions." The idea is to enjoy your coffee with some earthy, herbal notes. Then there\'s cold brew, known for being a "smooth, low-acid base" because it\'s extracted slowly. So, while neither is explicitly called "healthy" in the medical sense here, those are the closest descriptions provided!'

In [50]:
# We want to add reank module after the retrival
## Re-rank function
import numpy as np

def rerank_docs_cosine(results, query):
    ## Custom re-ranking methodology
    q = model_emb.encode([query], normalize_embeddings=True)[0]
    # very light re-score by cosine(q, preview embedding)
    rescored = []
    for r in results:
        emb = model_emb.encode([r["preview"]], normalize_embeddings=True)[0]
        score2 = float(np.dot(q, emb))
        rescored.append({**r, "score2": score2})
    return sorted(rescored, key=lambda x: x["score2"], reverse=True)

In [53]:
# Basic RAG
from langchain_core.runnables import RunnableLambda

chain = (
    RunnableLambda(hybrid_query) | RunnableLambda(lambda results:rerank_docs_cosine(results, user_q))
)
user_q = "Tell me about healthy coffee"

chain.invoke(user_q)

[{'id': '82f6dbe8-07d3-4ec9-b63b-f67468a088f6',
  'score': 0.8636,
  'preview': 'Overview Cold brew extracts coffee slowly at room temperature or in the fridge, yielding a smooth, low-acid base. Indian spices can be introduced via syrups or short infusions after brewing. Ingredients (Base & Optional) Coarsely ground coffee (1:8–1:10 coffee to water by weight) Cold water Spice syrup (cardamom-cinnamon-clove) optional Milk/ice as desired Brew 12–18 hours; strain thoroughly. Spices can be overpowering if steeped too long—add via syrup or brief infusion. Method',
  'score2': 0.4908428192138672},
 {'id': 'c9c3edfc-e9cf-433f-99dc-763faba0c5b9',
  'score': 0.9073,
  'preview': 'Beaten Coffee (Phenti Hui Coffee) Updated on August 17, 2025 Disclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines. Overv

In [56]:
# Basic RAG
from langchain_core.runnables import RunnableLambda

chain = (
      RunnableLambda(hybrid_query) # Intital retrival
    | RunnableLambda(lambda results:rerank_docs_cosine(results, user_q)) # Re-ranked retirval
    | RunnableLambda(lambda re_rank_results:create_prompt(user_q, re_rank_results)) #Creating prompt
    | llm
    | StrOutputParser()
)
user_q = "Tell me about healthy coffee"

chain.invoke(user_q)

'Hey there! So, the sources don\'t explicitly call any of these coffees "healthy," but they do offer a few interesting angles! Cold brew, for instance, is described as a "smooth, low-acid base," which some folks might prefer. Then there\'s ashwagandha coffee, which mixes coffee with ashwagandha, an herb "widely used in Indian traditions." Plus, many of these preparations feature Indian spices like cardamom and cinnamon, tapping into long-standing "Indian culinary practice." Just a friendly heads-up from the text: it\'s always smart to chat with a professional before trying new herbs, as this info isn\'t medical advice.'

In [57]:
# Basic RAG
from langchain_core.runnables import RunnableLambda

chain = (
      RunnableLambda(hybrid_query) # Intital retrival
    | RunnableLambda(lambda results:rerank_docs_cosine(results, user_q)) # Re-ranked retirval
    | RunnableLambda(lambda re_rank_results:create_prompt(user_q, re_rank_results)) #Creating prompt
    | llm
    | StrOutputParser()
)
user_q2 = "can you compare it with non-healthy coffee"

chain.invoke(user_q2)

'Hey! So, when it comes to "healthy coffee" based on these sources, it\'s interesting. You\'ve got Ashwagandha coffee, which mixes coffee with ashwagandha powder, an herb "widely used in Indian traditions." It\'s even called an "Adaptogenic Latte," aiming for earthy, herbal notes. Then there\'s Turmeric coffee, a "Haldi Cappuccino," incorporating turmeric, an ingredient with a "long culinary presence," giving it a warm, earthy flavor.\n\nBoth are described more from a culinary and cultural angle, focusing on ingredients and taste. The sources are pretty clear they\'re sharing general info and "not medical advice," so they don\'t actually go into specific health benefits. They even advise consulting a professional before trying new routines, especially if you have health conditions.'

In [ ]:
def check_len(user_q):
    ## Present
    # checking leng
    # passing new llm model 
    # getting updated prompt
    return user_q

In [ ]:
# Checking the questino, if the len is less than 100 words pass it another llm and get min 100 len of words
# is len is more then directly pass
from langchain_core.runnables import RunnableLambda

chain = (
    RunnableLambda(check_len)
    | RunnableLambda(hybrid_query) # Intital retrival
    | RunnableLambda(lambda results:rerank_docs_cosine(results, user_q)) # Re-ranked retirval
    | RunnableLambda(lambda re_rank_results:create_prompt(user_q, re_rank_results)) #Creating prompt
    | llm
    | StrOutputParser()
)
user_q2 = "can you compare it with non-healthy coffee"

chain.invoke(user_q2)

# Insertion into Vector Using LangChain

Steps for Insertion

Create vector index --> Load data --> Chunking --> Load Embedding Model--> Embeddings generation --> Organize & Upsert into Vector DB

In [20]:
# pip install langchain-community langchain-huggingface sentence-transformers pinecone-client beautifulsoup4 scikit-learn

from pathlib import Path
from typing import List, Dict, Any
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import BSHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableLambda

from sklearn.feature_extraction.text import TfidfVectorizer
from pinecone import Pinecone, ServerlessSpec

import os

In [22]:
# Creating Index
INDEX_NAME = "vector-hybrid-v2"

# ---------- Step 1–3: Initialize Pinecone, ensure index, get handle ----------
pc = Pinecone(api_key=pinecone_api_key)

In [23]:
# ---------- config ----------
if INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,            # must match dense model
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(INDEX_NAME)

In [24]:
index

## Loading the data 

In [26]:
#!pip install unstructured
from langchain_community.document_loaders import DirectoryLoader, UnstructuredHTMLLoader

# Load all HTML files from a folder into LangChain Documents
loader = DirectoryLoader(
    "coffee_pages",
    glob="**/*.html",
    loader_cls=UnstructuredHTMLLoader
)
docs = loader.load()
print("Loaded:", len(docs))

Loaded: 15


In [28]:
docs[0]

Document(metadata={'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html'}, page_content='Ashwagandha Coffee (Adaptogenic Latte)\n\nUpdated on August 17, 2025\n\nDisclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines.\n\nOverview\n\nAshwagandha coffee combines roasted coffee with powdered ashwagandha, an herb widely used in Indian traditions. The goal is to enjoy a familiar cup while layering in earthy, slightly bitter herbal notes. Home cooks typically treat this like a latte with spices for balance.\n\nIngredients (Base & Optional)\n\n1 cup milk of choice (dairy or plant-based)\n\n½–1 tsp instant coffee or a single espresso shot\n\n¼–½ tsp ashwagandha powder (culinary grade)\n\n¼ tsp cinnamon or cardamom (optional)\n\nSweetener to taste (jaggery, honey, or sugar)\n\nA pinch

## Chucking the data

In [30]:
#!pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 68


In [32]:
chunks[0]

Document(metadata={'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html'}, page_content='Ashwagandha Coffee (Adaptogenic Latte)\n\nUpdated on August 17, 2025\n\nDisclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines.\n\nOverview')

## Creating Embeddings

In [35]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# MiniLM-L6-v2 → 384-dim
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)

C:\Users\Harsha\AppData\Local\Temp\ipykernel_18364\306086346.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [36]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
), model_name='sentence-transformers/paraphrase-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [37]:
## Creating embeddings for Chunks & Organizing & Upsert

In [38]:
from langchain_pinecone import PineconeVectorStore

vectorstore_from_docs = PineconeVectorStore.from_documents(
    chunks,      # your split LangChain Documents
    embedding=embedding,    # HF embedder (384)
    index_name=INDEX_NAME, 
    text_key="text",
)

# Insertion into Vector db2 Using custome function via Runnable chains

Steps for Insertion

Create vector index --> Load data --> Chunking --> Load Embedding Model--> Embeddings generation --> Organize & Upsert into Vector DB

In [41]:
# Create dense + sparse encoders and encode corpus
import joblib
import pickle
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2" 

def step5_encode(payload: Dict[str, Any]) -> Dict[str, Any]:
    chunks = payload["chunks"]
    corpus = [c.page_content for c in chunks]

    # dense via HuggingFaceEmbeddings (normalized)
    embed = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        encode_kwargs={"normalize_embeddings": True}
    )
    dense_vectors = embed.embed_documents(corpus)

    # sparse via TF-IDF (fit on corpus)
    vectorizer = TfidfVectorizer(
        lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=1
    )
    tfidf_matrix = vectorizer.fit_transform(corpus)

    return {
        "chunks": chunks,
        "dense_vectors": dense_vectors,
        "tfidf_matrix": tfidf_matrix,
    }

In [47]:
data = {"chunks": chunks}
#step5_encode(data)

In [48]:
def csr_row_to_pinecone_sparse(csr_row) -> Dict[str, List[float]]:
    coo = csr_row.tocoo()
    return {"indices": coo.col.tolist(), "values": coo.data.astype(float).tolist()}
    
#Package for Pinecone and upsert
def step6_package_and_upsert(payload: Dict[str, Any]) -> Dict[str, Any]:
    chunks = payload["chunks"]
    dense_vectors = payload["dense_vectors"]
    tfidf_matrix = payload["tfidf_matrix"]

    vectors = []
    for i, (c, dense) in enumerate(zip(chunks, dense_vectors)):
        sparse = csr_row_to_pinecone_sparse(tfidf_matrix[i])
        parent = c.metadata.get("doc_id", f"doc{abs(hash(c.metadata.get('path', ''))) & 0xffff}")
        vid = f"{parent}::chunk_{i:04d}"
        vectors.append({
            "id": vid,
            "values": dense,
            "sparse_values": sparse,
            "metadata": {
                "text": c.page_content,
                **{k: v for k, v in c.metadata.items() if k != "text"}
            }
        })

    index.upsert(vectors=vectors)
    return {"upserted": len(vectors), "index": INDEX_NAME}
    # return vectors

In [49]:
from langchain_core.runnables import RunnableLambda

chain = (
      RunnableLambda(get_chunks) ## Creating chunks for the docs
    | RunnableLambda(step5_encode) ## Creating emebeddings sparse & dense for chunks
    | RunnableLambda(step6_package_and_upsert)  ## Upserting
)

chain.invoke(docs)

{'chunks': [Document(metadata={'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html', 'text': 'Ashwagandha Coffee (Adaptogenic Latte)\n\nUpdated on August 17, 2025\n\nDisclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines.\n\nOverview'}, page_content='Ashwagandha Coffee (Adaptogenic Latte)\n\nUpdated on August 17, 2025\n\nDisclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines.\n\nOverview'),
  Document(metadata={'source': 'coffee_pages\\01_ashwagandha_coffee_adaptogenic_latte.html', 'text': 'Overview\n\nAshwagandha coffee combines roasted coffee with powdered ashwagandha, an herb 